1. Import Libraries and Load Your Curated Dataset

In [0]:
from pyspark.sql import SparkSession
import pandas as pd


spark = SparkSession.builder.getOrCreate()

# Load curated dataset
df_spark = spark.table("workspace.silver.labeled_step_test")


# Convert to pandas
df = df_spark.toPandas()
df.head()



2. Define Your Feature Columns

In [0]:
feature_cols_numeric = ["distance_cm"]
feature_cols_categorical = ["sensor_type", "device_id"]
label_col = "step_label"



3. Create a Train/Test Split

In [0]:
from sklearn.model_selection import train_test_split

X = df[feature_cols_numeric + feature_cols_categorical]
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



4. 
Build Preprocessing Steps

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, feature_cols_numeric),
        ("cat", categorical_transformer, feature_cols_categorical)
    ]
)



5. Build a Scikit-Learn Pipeline


In [0]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor)
])



6. Fit the Pipeline and Transform the Data

In [0]:
pipeline.fit(X_train)

X_train_transformed = pipeline.transform(X_train)
X_test_transformed = pipeline.transform(X_test)


7. Save Your Processed Feature Set and Pipeline

In [0]:
import os
import joblib

base_path = "./etl_pipeline"
os.makedirs(base_path, exist_ok=True)

joblib.dump(pipeline, f"{base_path}/stedi_feature_pipeline.pkl")
joblib.dump(X_train_transformed, f"{base_path}/X_train_transformed.pkl")
joblib.dump(X_test_transformed, f"{base_path}/X_test_transformed.pkl")
joblib.dump(y_test, f"{base_path}/y_test.pkl")
joblib.dump(y_train, f"{base_path}/y_train.pkl")


Ethics Reflection: 
Using a consistent and reproducible feature pipeline helps prevent unfairness and hidden bias by ensuring that all data is processed in the same way every time a model is trained or evaluated. This consistency reduces the risk of introducing unintended advantages or errors that could affect certain groups differently. A spiritual principle that helps me understand this importance is Doctrine and Covenants 130:20–21, which teaches that blessings are tied to consistent and dependable laws. Just as spiritual growth depends on reliable principles, fairness in machine learning depends on consistent and reproducible processes.
